# AgentWonder v2.5 — Live LLM Integration Test

This notebook demonstrates Version 2.5 features:

1. **Structured Logging** — JSON-formatted log output
2. **File-backed Persistence** — sessions and state persist to disk
3. **Real Gemini LLM calls** — replaces stub responses
4. **LLM-backed tools** — summarizer and classifier
5. **Full workflow execution** with real LLM output

## Setup

Set your Google API key in the cell below, then run all cells.
If no API key is set, the system falls back to stub responses.

In [1]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=Path(".env"))
    print("Loaded .env from repo root")
except ImportError:
    print("python-dotenv not installed; using existing environment variables")

# ============================================================
# SET YOUR GOOGLE API KEY HERE (or use a .env file in repo root)
# ============================================================
# os.environ["GOOGLE_API_KEY"] = "your-key-here"

if os.environ.get("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY loaded from environment")
else:
    print("GOOGLE_API_KEY not set; notebook will use stub LLM responses")

# To run in stub mode (no real LLM calls), leave the key unset.

Loaded .env from repo root
GOOGLE_API_KEY loaded from environment


## 1. Initialize — Logging, Registries, Persistence

In [2]:
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

os.chdir(repo_root)

from agentwonder.logging import configure_logging, get_logger
from agentwonder.registry import TemplateRegistry, ToolRegistry, PromptRegistry, PolicyRegistry
from agentwonder.runtime.llm_client import GeminiClient
from agentwonder.runtime.session_store import FileSessionStore
from agentwonder.runtime.state_store import FileStateStore
from agentwonder.runtime.executor import WorkflowExecutor
from agentwonder.compiler.loader import load_yaml
from agentwonder.compiler.validators import validate_workflow, cross_validate_workflow
from agentwonder.compiler.resolver import resolve_workflow
from agentwonder.compiler.builder import build_plan
from agentwonder.schemas.run import RunRequest
import json, tempfile

# Configure structured logging
configure_logging(level="INFO", json_output=False)
logger = get_logger("notebook")

# Load registries
config = Path("config")
tr = TemplateRegistry(); tr.load_from_directory(config / "templates")
tl = ToolRegistry(); tl.load_from_directory(config / "tools")
pr = PromptRegistry(); pr.load_from_directory(config / "prompts")
po = PolicyRegistry(); po.load_from_directory(config / "policies")

tools_d = {t.id: t for t in tl.list_all()}
templates_d = {t.id: t for t in tr.list_all()}
prompts_d = {t.id: t for t in pr.list_all()}
policies_d = {t.id: t for t in po.list_all()}

# Initialize Gemini client
llm = GeminiClient()
print(f"LLM configured: {llm.is_configured}")
print(f"Templates: {len(templates_d)}, Tools: {len(tools_d)}")

# Set up file-backed persistence in a temp directory
data_dir = Path(tempfile.mkdtemp(prefix="agentwonder_"))
session_store = FileSessionStore(data_dir=data_dir)
state_store = FileStateStore(data_dir=data_dir)
executor = WorkflowExecutor(
    session_store=session_store,
    state_store=state_store,
    llm_client=llm,
)
print(f"Persistence: file-backed at {data_dir}")

def pipeline(wf_file, inputs):
    raw = load_yaml(config / "workflows" / wf_file)
    wf = validate_workflow(raw)
    errors = cross_validate_workflow(wf, tools_d, templates_d, policies_d, prompts_d)
    assert not errors, f"Errors: {errors}"
    resolved = resolve_workflow(wf, tools_d, templates_d, prompts_d, policies_d)
    return build_plan(resolved), RunRequest(workflow_id=wf.id, inputs=inputs)

10:38:45 [   INFO] agentwonder.compiler.loader: Loaded 5 YAML files from config/templates
10:38:45 [   INFO] agentwonder.registry.templates: Registered template 'evaluator_loop' v1.0.0
10:38:45 [   INFO] agentwonder.registry.templates: Registered template 'parallel_fanout_aggregate' v1.0.0
10:38:45 [   INFO] agentwonder.registry.templates: Registered template 'router_specialists' v1.0.0
10:38:45 [   INFO] agentwonder.registry.templates: Registered template 'sequential_with_approval' v1.0.0
10:38:45 [   INFO] agentwonder.registry.templates: Registered template 'single_agent_with_tools' v1.0.0
10:38:45 [   INFO] agentwonder.compiler.loader: Loaded 13 YAML files from config/tools
10:38:45 [   INFO] agentwonder.registry.tools: Registered tool 'classify_ticket' v1.0.0
10:38:45 [   INFO] agentwonder.registry.tools: Registered tool 'fetch_bonds' v1.0.0
10:38:45 [   INFO] agentwonder.registry.tools: Registered tool 'fetch_break_details' v1.0.0
10:38:45 [   INFO] agentwonder.registry.tools: Reg

LLM configured: True
Templates: 5, Tools: 13
Persistence: file-backed at /var/folders/tm/_751k55s6xz78x_1pjy90hpw0000gn/T/agentwonder_sffy_7pa


## 2. LLM-Backed Tools — Summarizer and Classifier

In [3]:
from agentwonder.tools.llm_tools import LLMSummarizer, LLMClassifier

summarizer = LLMSummarizer(client=llm)
classifier = LLMClassifier(client=llm)

# Test summarizer
summary_result = await summarizer.call({
    "text": (
        "AgentWonder is a YAML-first agentic workflow platform built on Google ADK. "
        "It uses Pydantic v2 for schema validation, supports five workflow templates "
        "(sequential, router, parallel, evaluator loop, single agent), and provides "
        "governance features including approval gates, tool registries, and observability. "
        "Version 2.5 adds real Gemini LLM integration, structured logging, and "
        "pluggable file-backed persistence."
    ),
    "max_length": "2 sentences",
})
print("=== Summarizer ===")
print(f"Status: {summary_result['status']}")
print(f"Summary: {summary_result['summary']}\n")

# Test classifier
classify_result = await classifier.call({
    "text": "My account was charged twice for the same transaction last month",
    "categories": ["billing", "technical_support", "account_access", "general_inquiry"],
})
print("=== Classifier ===")
print(f"Status: {classify_result['status']}")
print(f"Category: {classify_result['category']}")
print(f"Confidence: {classify_result['confidence']}")
print(f"Reasoning: {classify_result['reasoning']}")

10:38:49 [   INFO] agentwonder.tools.llm_tools: llm_summarizer invoked text_len=412
10:38:49 [   INFO] agentwonder.runtime.llm_client: generating llm response model=gemini-2.0-flash prompt_len=503
10:38:49 [   INFO] google_genai.models: AFC is enabled with max remote calls: 10.
10:38:50 [   INFO] httpx: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 404 Not Found"
10:38:50 [  ERROR] agentwonder.runtime.llm_client: llm call failed model=gemini-2.0-flash error=404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

## 3. Workflow Execution with LLM — Sequential with Approval

In [ ]:
plan, req = pipeline("break_resolution_v1.yaml", {"break_id": "BRK-500", "source_system": "operations"})
status = await executor.execute(req, plan)

print(f"Run: {status.run_id}")
print(f"State: {status.state.value}")
print(f"Steps completed: {list(status.outputs.keys())}\n")

# Show the LLM agent step output (propose_resolution)
propose = status.outputs.get("propose_resolution", {})
print("=== propose_resolution output ===")
print(f"Model: {propose.get('model', 'n/a')}")
output_text = propose.get("output", "")
print(f"Output ({len(output_text)} chars):")
print(output_text[:500] if len(output_text) > 500 else output_text)

12:55:46 [   INFO] agentwonder.compiler.resolver: Resolved workflow 'break_resolution_v1': 4 tools, 1 prompts, 1 policies
12:55:46 [   INFO] agentwonder.compiler.builder: Built RuntimePlan for 'break_resolution_v1' v1.0.0: 5 steps, 5 parallel groups, 2 approval gates
12:55:46 [   INFO] agentwonder.runtime.executor: tool_call step step_id=enrich_break tool_id=fetch_break_details tool_type=rest
12:55:46 [   INFO] agentwonder.runtime.executor: tool_call step step_id=enrich_trade_context tool_id=fetch_trade_context tool_type=rest
12:55:46 [   INFO] agentwonder.runtime.executor: llm_agent step step_id=propose_resolution model=gemini-2.5-flash
12:55:46 [   INFO] agentwonder.runtime.approvals: Registered approval '0d27bbe7f329' for run='76cb788cf271', step='checker_approval'
12:55:46 [   INFO] agentwonder.runtime.executor: approval gate — auto-approving step_id=checker_approval
12:55:46 [   INFO] agentwonder.runtime.approvals: Approval '0d27bbe7f329' decided: approved by system_auto_approve_v

Run: 76cb788cf271
State: completed
Steps completed: ['enrich_break', 'enrich_trade_context', 'propose_resolution', 'checker_approval', 'update_downstream']

=== propose_resolution output ===
Model: gemini-2.5-flash
Output (49 chars):
[Stub LLM response for step 'propose_resolution']


## 4. File-backed Persistence Verification

In [ ]:
import os

# Check persisted files
session_files = list((data_dir / "sessions").glob("*.json"))
state_files = list((data_dir / "state").glob("*.json"))
print(f"Persisted session files: {len(session_files)}")
print(f"Persisted state files: {len(state_files)}")

# Read back from a NEW store instance to prove persistence
from agentwonder.runtime.session_store import FileSessionStore as FSS2
from agentwonder.runtime.state_store import FileStateStore as FST2

fresh_session = FSS2(data_dir=data_dir)
fresh_state = FST2(data_dir=data_dir)

run_id = status.run_id
persisted_session = fresh_session.get_session(run_id)
persisted_state = fresh_state.get_all(run_id)

print(f"\nReloaded session for run {run_id}:")
print(f"  workflow_id: {persisted_session['data'].get('workflow_id')}")
print(f"  final_state: {persisted_session['data'].get('final_state')}")
print(f"  trace events: {len(persisted_session['data'].get('trace_events', []))}")
print(f"\nReloaded state keys: {list(persisted_state.keys())}")
assert persisted_session["data"]["final_state"] == "completed"
print("\nPersistence verified!")

12:55:46 [   INFO] agentwonder.runtime.session_store: file session store initialized path=/var/folders/tm/_751k55s6xz78x_1pjy90hpw0000gn/T/agentwonder_0f0wuk9b/sessions
12:55:46 [   INFO] agentwonder.runtime.state_store: file state store initialized path=/var/folders/tm/_751k55s6xz78x_1pjy90hpw0000gn/T/agentwonder_0f0wuk9b/state


Persisted session files: 1
Persisted state files: 1

Reloaded session for run 76cb788cf271:
  workflow_id: break_resolution_v1
  final_state: completed
  trace events: 18

Reloaded state keys: ['enrich_break', 'enrich_trade_context', 'propose_resolution', 'checker_approval', 'update_downstream']

Persistence verified!


## 5. Summary

In [ ]:
print("AgentWonder v2.5 Validation Summary")
print("=" * 40)
print(f"LLM integration:    {'LIVE (Gemini)' if llm.is_configured else 'STUB mode'}")
print(f"Structured logging: OK")
print(f"File persistence:   OK ({len(session_files)} sessions, {len(state_files)} states)")
print(f"LLM summarizer:     {summary_result['status']}")
print(f"LLM classifier:     {classify_result['status']}")
print(f"Workflow execution: {status.state.value}")
print(f"Persistence reload: verified")
print()
print("All v2.5 features operational.")

AgentWonder v2.5 Validation Summary
LLM integration:    STUB mode
Structured logging: OK
File persistence:   OK (1 sessions, 1 states)
LLM summarizer:     success
LLM classifier:     success
Workflow execution: completed
Persistence reload: verified

All v2.5 features operational.
